In [1]:
import pandas as pd 
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import  StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.utils import resample
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.tree import DecisionTreeClassifier

In [45]:
data = pd.read_csv("Default.csv")
data.head()

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138950
3,No,No,529.250605,35704.493940
4,No,No,785.655883,38463.495880


# Accuracy Logistic Regression

## ACCURACY

In [46]:
data['default'] = data['default'].map({'No':0, 'Yes':1})

In [47]:
data = pd.get_dummies(data, columns=['student'], drop_first=True)

x =  data.drop("default", axis=1)
y =  data['default']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(x)


In [48]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

log_model = LogisticRegression(max_iter=1000, multi_class='ovr')
log_scores = cross_val_score(log_model, X_scaled, y, cv=kf, scoring='accuracy')

In [49]:
ACC_LOG = np.mean(log_scores)
print("\n🔹 Resultado de Regresión Logística:")
print(f"ACCURACY promedio: {np.mean(ACC_LOG):.4f}")


🔹 Resultado de Regresión Logística:
ACCURACY promedio: 0.9733


## AUC

In [25]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

log_model = LogisticRegression(max_iter=1000, multi_class='ovr')
log_scores = cross_val_score(log_model, X_scaled, y, cv=kf, scoring='roc_auc')

In [26]:
print("\n🔹 Resultado de Regresión Logística:")
print(f"AUC promedio: {np.mean(log_scores):.4f}")


🔹 Resultado de Regresión Logística:
AUC promedio: 0.9491


# ACCURACY DE BAGGING

Primero había hecho este código, pero luego cuando fuimos a hablar con usted al término de la clase nos dijo la parte de que la predicción va hasta el final, puesto que si lo metemos dentro del mismo for haría que las predicciones no se pudiesen comparar entre sí. (No había entendido muy bien eso pero creo tener la idea).

In [61]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils import resample
import numpy as np


data = pd.read_csv("Default.csv")
np.random.seed(0)

N_BOOT = 5000
accuracies_bag = []

data['default'] = data['default'].map({'No':0, 'Yes':1})
data = pd.get_dummies(data, columns=['student'], drop_first=True)

X = data.drop("default", axis=1).values
y = data["default"].values

for _ in range(N_BOOT):

    # 1. Bootstrap sample
    Xb, yb = resample(X, y, replace=True)

    # 2. Selección aleatoria de 2 columnas
    cols = np.random.choice(X.shape[1], size=2, replace=False)

    Xb_sub = Xb[:, cols]         # para entrenar
    X_sub  = X[:, cols]          # para evaluar

    # 3. Entrenar árbol
    model = DecisionTreeClassifier()
    model.fit(Xb_sub, yb)

    # 4. Predecir sobre TODO el dataset original
    preds = model.predict(X_sub)

    acc = (preds == y).mean()
    accuracies_bag.append(acc)

print("Accuracy Bagging:", np.mean(accuracies_bag))


Accuracy Bagging: 0.98159106


In [58]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils import resample
import numpy as np
import pandas as pd
from scipy.stats import mode

data = pd.read_csv("Default.csv")
np.random.seed(0)

# Preprocesamiento
data['default'] = data['default'].map({'No':0, 'Yes':1})
data = pd.get_dummies(data, columns=['student'], drop_first=True)

X = data.drop("default", axis=1).values
y = data["default"].values

N_BOOT = 5000
pred_matrix = []   # aquí se guardan las predicciones

for _ in range(N_BOOT):
    # primero hacemos el bootstrap
    Xb, yb = resample(X, y, replace=True)

    # Luego se seleccionan 2 columnas aleatorias
    cols = np.random.choice(X.shape[1], size=2, replace=False)

    Xb_sub = Xb[:, cols]  # este nsos sirv para entrenar, considerando esas 2 columnas
    X_sub  = X[:, cols]   # y este para predecir, igual con las mismas columnas

    # 3. Se entrena el árbol
    model = DecisionTreeClassifier(max_depth=3)
    model.fit(Xb_sub, yb)

    # 4. Vamos almacenando la predicción completa del dataset original.
    preds = model.predict(X_sub)
    pred_matrix.append(preds)

# Convertimos la predicción a tamaño [N_BOOT, n_muestras] a matriz
pred_matrix = np.array(pred_matrix)

# 5. Tomamos la moda por fila (por observación)
final_preds = mode(pred_matrix, axis=0).mode.flatten()

# 6. Accuracy final
accuracy = (final_preds == y).mean()

print("Accuracy Bagging:", accuracy)


C:\Users\Admin\AppData\Local\Temp\ipykernel_9660\3707757788.py:43: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  final_preds = mode(pred_matrix, axis=0).mode.flatten()


Accuracy Bagging: 0.9734


In [59]:
ACC_bag = accuracy

In [60]:
ACC_bag, ACC_LOG

(0.9734, 0.9733)

Aquí tenemos nuestras dos métricas, las cuales son muy parecidas. Vemos que con el uso de bagging sí se mejoró el ACCURACY pero tampoco por mucho.
Analizando el data set, los procedimientos y resultados, llegué a la misma conclusión que cuando estaba optimizando mis valores en mi proyecto pasado. Noto que no hubo mucha mejoría, pero aún así se está mejorando algo y eso es lo que realmente importa. Además, tiene mucho que ver por el data set que tenemos, puesto que es extremadamente simple y el hecho de meterle modelos como el de árboles o bootstrapping no hará que mejore tanto porque a lo que puede llegar una bagging  (en este caso) también lo alcanza a cubrir la regresión logística. 
Además, en el data set que se tiene, la variable balance explica casi a la perfección la variable predicitiva default. Por lo tanto lo que un modelo de bagging puede llegar a hacer, en este caso, logistic regresion también alcanza a cubrir esta parte, por ello la cercanía de sus métricas y por ello que no mejora tanto.

Es importante mencionar que en la parte del modelo de árbol le acorté la profundidad puesto que de no hacerlo así, la métrica me estaba arrojando 1 siempre. Por ello, investigando un poco, comprendí que en el modelo de arboles de regresión es mejor agregarle este acotamiento para evitar que memorice por completo los datos, y por lo tanto reducir las posibilidades de obtener un overfitting que es justo lo que me estaba pasando. Así que lo acoté a 3 y así se redujo el ACC. Esto se logra porque las divisiones o ramas del árbol se reducen y evita que aprenda patrones muy específicos. Así solo se centra como en características generales y evita el sobre ajuste.